<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed, test_size=0.2, sample_size=10000):
    X, y = fetch_openml(
        "Fashion-MNIST",
        version=1,
        as_frame=False,
        return_X_y=True,
        parser="liac-arff",
        data_home="./.sklearn_data",
    )
    y = y.astype(int)

    if sample_size is not None:
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=sample_size,
            stratify=y,
            random_state=123,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )
    return X_train, X_test, y_train, y_test


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed):
    base_tree = DecisionTreeClassifier(max_depth=1, random_state=seed)
    model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=100,
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)


Nao e necessario normalizar os dados para `RandomForestClassifier` e `AdaBoostClassifier` com arvores como base, porque eles tomam decisoes por cortes nas features e nao por distancia entre vetores.

Para deixar a execucao rapida e reprodutivel, os resultados abaixo foram calculados com uma amostra estratificada de `10.000` imagens do Fashion-MNIST.


# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc


Aqui o foco nao e profundidade de uma unica arvore, e sim comparar modelos de ensemble de forma reproduzivel e interpretar os resultados obtidos.


# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="macro"),
        "recall": recall_score(y_test, y_pred, average="macro"),
        "f1": f1_score(y_test, y_pred, average="macro"),
    }

X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

rf_metrics = compute_metrics(rf_model, X_test, y_test)
ab_metrics = compute_metrics(ab_model, X_test, y_test)

print("Random Forest:", rf_metrics)
print("AdaBoost:", ab_metrics)


Resultados obtidos com `seed=42`:

- Random Forest: acuracia `0.8525`, precisao `0.8520`, recall `0.8525`, F1-score `0.8502`
- AdaBoost: acuracia `0.5110`, precisao `0.4881`, recall `0.5110`, F1-score `0.4592`

O melhor desempenho inicial foi do `Random Forest`, com vantagem ampla em todas as metricas.


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [ ]:
for seed in [42, 7]:
    rf_acc = run_pipeline(model_type="rf", seed=seed)
    ab_acc = run_pipeline(model_type="ab", seed=seed)
    print(f"Seed = {seed}")
    print(f"  Random Forest acc: {rf_acc:.4f}")
    print(f"  AdaBoost acc: {ab_acc:.4f}")


Com seeds diferentes, os resultados mudaram um pouco:

- `seed=42`: Random Forest `0.8525`, AdaBoost `0.5110`
- `seed=7`: Random Forest `0.8445`, AdaBoost `0.5120`

O experimento continua sendo reprodutivel porque a mesma seed reproduz os mesmos resultados. O que mudou foi a amostra treino/teste gerada por seeds diferentes, e isso alterou levemente a acuracia.


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

train_acc_rf = evaluate(rf_model, X_train, y_train)
test_acc_rf = evaluate(rf_model, X_test, y_test)

train_acc_ab = evaluate(ab_model, X_train, y_train)
test_acc_ab = evaluate(ab_model, X_test, y_test)

print(f"Random Forest - treino: {train_acc_rf:.4f}")
print(f"Random Forest - teste: {test_acc_rf:.4f}")
print(f"AdaBoost - treino: {train_acc_ab:.4f}")
print(f"AdaBoost - teste: {test_acc_ab:.4f}")


Comparando treino e teste com `seed=42`:

- Random Forest: treino `1.0000`, teste `0.8525`
- AdaBoost: treino `0.5316`, teste `0.5110`

Existe overfitting claro no `Random Forest`, porque ele memoriza muito melhor o treino do que generaliza no teste. O `AdaBoost` ficou bem mais fraco no geral, mas com diferenca menor entre treino e teste.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = load_data(seed=42)

print("Random Forest")
for n_estimators in [50, 100, 200]:
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)
    rf_acc = evaluate(rf_model, X_test, y_test)
    print(f"  n_estimators={n_estimators}: {rf_acc:.4f}")

print("\nAdaBoost")
for n_estimators in [50, 100, 200]:
    ab_model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=n_estimators,
        random_state=42,
    )
    ab_model.fit(X_train, y_train)
    ab_acc = evaluate(ab_model, X_test, y_test)
    print(f"  n_estimators={n_estimators}: {ab_acc:.4f}")


Impacto de `n_estimators`:

- Random Forest: `50 -> 0.8495`, `100 -> 0.8525`, `200 -> 0.8545`
- AdaBoost: `50 -> 0.5615`, `100 -> 0.5110`, `200 -> 0.6220`

No `Random Forest`, o ganho foi pequeno e estavel. No `AdaBoost`, a variacao foi bem maior, entao ele se mostrou mais sensivel a esse hiperparametro neste experimento.


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

1. A acuracia nao e suficiente sozinha para avaliar os modelos, porque ela resume tudo em um unico numero e pode esconder erros importantes entre classes. Por isso, precisao, recall e F1-score ajudam a mostrar melhor o comportamento do classificador.

2. Para reduzir a chance de o resultado ocorrer por acaso, usamos divisao estratificada, controle de seed e repeticao com seeds diferentes. Isso nao elimina toda variacao, mas mostra se o desempenho se mantem relativamente estavel.

3. Dois problemas metodologicos possiveis sao: usar apenas uma divisao treino/teste em vez de validacao cruzada, e comparar modelos sem explorar hiperparametros com o mesmo cuidado para ambos. Isso pode favorecer um modelo injustamente.

4. O pipeline implementado e confiavel como base, porque separa carregamento, treino e avaliacao de forma reproduzivel. Mesmo assim, ele pode ser melhorado com validacao cruzada, matriz de confusao e uma busca mais sistematica de hiperparametros.
